# Affinity-guided LIVECell training

This Kaggle notebook installs the repository dependencies, clones the public model repository, prepares the attached dataset as paired image/mask directories, trains on the configured GPU, and copies `runs/` to the Kaggle output root.

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile

REPOSITORY_URL = 'https://github.com/EugenTheMachine/some_model.git'
REPOSITORY_DIR = Path('/kaggle/working/some_model_repo')
DATASET_SOURCE = 'eugenfromkharkov/livecell-cellseg1-a172'
DATASET_SLUG = DATASET_SOURCE.split('/', maxsplit=1)[1]
INPUT_ROOT = Path('/kaggle/input')
SUPPORTED_IMAGE_EXTENSIONS = {'.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp'}

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(
    ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(REPOSITORY_DIR)],
    check=True,
)
print(f'Cloned model source into {REPOSITORY_DIR}')

In [ ]:
requirements_path = REPOSITORY_DIR / 'requirements.txt'
module_for_package = {
    'numpy': 'numpy',
    'opencv-python-headless': 'cv2',
    'kornia': 'kornia',
    'monai': 'monai',
    'Pillow': 'PIL',
    'PyYAML': 'yaml',
    'scikit-image': 'skimage',
    'scipy': 'scipy',
    'tifffile': 'tifffile',
    'torch': 'torch',
    'torchstain': 'torchstain',
    'torchvision': 'torchvision',
    'tqdm': 'tqdm',
}
missing_packages = []
for raw_line in requirements_path.read_text(encoding='utf-8').splitlines():
    package = raw_line.strip()
    if not package or package.startswith('#'):
        continue
    module_name = module_for_package.get(package)
    if module_name is None:
        raise ValueError(f'No import check is configured for dependency {package!r}')
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package)
if missing_packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages],
        check=True,
    )
else:
    print('All declared dependencies are already installed.')
sys.path.insert(0, str(REPOSITORY_DIR / 'src'))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is unavailable; this kernel requires a T4 CUDA device.')
GPU_NAME = torch.cuda.get_device_name(0)
if 'T4' not in GPU_NAME.upper():
    raise RuntimeError(f'Expected a Tesla T4 GPU, found {GPU_NAME!r}.')
print(f'Using Kaggle GPU: {GPU_NAME}')

In [ ]:
dataset_candidates = [INPUT_ROOT / DATASET_SLUG]
if INPUT_ROOT.is_dir():
    dataset_candidates.extend(
        candidate
        for candidate in INPUT_ROOT.iterdir()
        if candidate.is_dir() and DATASET_SLUG.lower() in candidate.name.lower()
    )
DATASET_ROOT = next((path for path in dataset_candidates if path.is_dir()), None)
if DATASET_ROOT is None:
    available = sorted(path.name for path in INPUT_ROOT.iterdir()) if INPUT_ROOT.is_dir() else []
    raise FileNotFoundError(
        f'Dataset {DATASET_SOURCE!r} is not mounted under {INPUT_ROOT}. '
        f'Available inputs: {available}'
    )

def image_mask_pairs(image_dir, mask_dir):
    images = sorted(
        path for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )
    masks = sorted(
        path for path in mask_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )
    by_name = {path.name: path for path in masks}
    by_stem = {path.stem: path for path in masks}
    pairs = []
    unmatched = []
    for image_path in images:
        mask_path = (
            by_name.get(image_path.name)
            or by_stem.get(image_path.stem)
            or by_name.get(f'mask_{image_path.stem}.png')
        )
        if mask_path is None:
            unmatched.append(image_path)
        else:
            pairs.append((image_path, mask_path))
    if unmatched and len(images) == len(masks):
        paired_images = {image for image, _ in pairs}
        paired_masks = {mask for _, mask in pairs}
        remaining_images = [path for path in images if path not in paired_images]
        remaining_masks = [path for path in masks if path not in paired_masks]
        pairs.extend(zip(remaining_images, remaining_masks))
    elif unmatched:
        raise ValueError(
            f'Could not pair {len(unmatched)} image(s) in {image_dir} with masks.'
        )
    return sorted(pairs, key=lambda pair: pair[0].name)

def split_name_for(image_dir):
    name = image_dir.parent.name.lower()
    if 'valid' in name or name in {'val', 'validation'}:
        return 'val'
    if 'test' in name:
        return 'test'
    return 'train'

serialized_datasets = sorted(DATASET_ROOT.rglob('*_dataset.pth'))
if not serialized_datasets:
    serialized_datasets = sorted(DATASET_ROOT.rglob('*.pth'))
if len(serialized_datasets) > 1:
    raise ValueError(f'Expected one serialized dataset in {DATASET_ROOT}, found {serialized_datasets}')
SERIALIZED_DATASET = serialized_datasets[0] if serialized_datasets else None
prepared_data_dir = Path(tempfile.mkdtemp(prefix='livecell-paired-data-'))
prepared_counts = {'train': 0, 'val': 0, 'test': 0}
pair_directories = []
for image_dir in sorted(DATASET_ROOT.rglob('images')):
    if not image_dir.is_dir():
        continue
    mask_dir = next(
        (image_dir.parent / name for name in ('masks', 'labels')
         if (image_dir.parent / name).is_dir()),
        None,
    )
    if mask_dir is not None:
        pair_directories.append((image_dir, mask_dir))

for image_dir, mask_dir in pair_directories:
    split = split_name_for(image_dir)
    pairs = image_mask_pairs(image_dir, mask_dir)
    if not pairs:
        continue
    target_images = prepared_data_dir / split / 'images'
    target_masks = prepared_data_dir / split / 'masks'
    target_images.mkdir(parents=True, exist_ok=True)
    target_masks.mkdir(parents=True, exist_ok=True)
    for image_path, mask_path in pairs:
        index = prepared_counts[split]
        stem = f'sample_{index:08d}'
        (target_images / f'{stem}{image_path.suffix.lower()}').symlink_to(image_path.resolve())
        (target_masks / f'{stem}{mask_path.suffix.lower()}').symlink_to(mask_path.resolve())
        prepared_counts[split] += 1

if not prepared_counts['train'] and SERIALIZED_DATASET is None:
    raise ValueError(
        f'No paired train/images and train/masks data found under {DATASET_ROOT}. '
        'The source may use annotations that need rasterization first.'
    )
print(f'Attached dataset: {DATASET_SOURCE} at {DATASET_ROOT}')
if SERIALIZED_DATASET is not None:
    print(f'Using serialized dataset: {SERIALIZED_DATASET}')
print(f'Preloaded dataset pairs by split: {prepared_counts}')
print(f'Prepared dataset directory: {prepared_data_dir}')

In [ ]:
from model import InstanSegModel

epochs = int(os.environ.get('KAGGLE_TRAIN_EPOCHS', '1'))
imgsz = int(os.environ.get('KAGGLE_TRAIN_IMGSZ', '256'))
train_data_ratio = float(os.environ.get('KAGGLE_TRAIN_DATA_RATIO', '1.0'))
if imgsz <= 0 or imgsz % 16:
    raise ValueError('KAGGLE_TRAIN_IMGSZ must be a positive multiple of 16.')

trainer = InstanSegModel(
    data_dir=str(DATASET_ROOT if SERIALIZED_DATASET is not None else prepared_data_dir),
    dataset=str(SERIALIZED_DATASET) if SERIALIZED_DATASET is not None else None,
    experiment_name='kaggle_livecell_ag_mlp',
    imgsz=imgsz,
    train_data_ratio=train_data_ratio,
    seed=42,
    dim_in=1,
    epochs=epochs,
    patience=2,
    batch_size=1,
    val_ratio=0.2,
    test_ratio=0.2,
    val_interval=1,
    warmup_epochs=0,
    resume=False,
    amp=torch.cuda.is_available(),
    num_workers=0,
    input_channels=1,
    backbone_channels=[16, 32, 64, 128, 256],
    feature_channels=16,
    num_slots=64,
    mlp_channels=[32, 16],
    use_aggregation=True,
)
print(f'Training on {GPU_NAME} for {epochs} epoch(s).')
history = trainer.train()
print(f'Completed {len(history)} epoch(s).')
print(f'Run artifacts: {REPOSITORY_DIR / "runs"}')

In [ ]:
OUTPUT_ROOT = Path('/kaggle/working')
RUNS_SOURCE = REPOSITORY_DIR / 'runs'
RUNS_OUTPUT = OUTPUT_ROOT / 'runs'
if not RUNS_SOURCE.is_dir():
    raise FileNotFoundError(f'Training artifacts were not created: {RUNS_SOURCE}')
if RUNS_OUTPUT.exists():
    shutil.rmtree(RUNS_OUTPUT)
shutil.copytree(RUNS_SOURCE, RUNS_OUTPUT)
if prepared_data_dir.exists():
    shutil.rmtree(prepared_data_dir)
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
print(f'Artifacts copied to {RUNS_OUTPUT}')
print(f'Cloned repository removed: {not REPOSITORY_DIR.exists()}')